In [ ]:
# Cell 222 (10_continuous_ref_and_rq3_ci.ipynb, cell 1): Phase 8 inspection. FRESH KERNEL. READ-ONLY, writes NOTHING.
def _c222():
    import os, re, ast, json, hashlib, pickle
    import pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sha = lambda p: hashlib.sha256(open(R + "/" + p, "rb").read()).hexdigest()[:12]
    G = {"masking": "c6094c2b232e", "rq3": "9ccbba66073b", "safe_save": "afff88e2b30a", "checks": "ef542907c037", "decision_value": "c1476cef61d1",
         "loglik": "cdc26c96a8d6", "em": "c245161bebeb", "sensitivity": "e49b368ee726", "structure": "effc1d6565f8"}
    for m, h in G.items(): print("guard %-15s %s %s" % (m, sha("shared/src/%s.py" % m), "OK" if sha("shared/src/%s.py" % m) == h else "MISMATCH: STOP"))
    print("fast_auc.py exists:", os.path.exists(R + "/shared/src/fast_auc.py"), "| notebook 10 exists:", os.path.exists(R + "/selection/notebooks/10_continuous_ref_and_rq3_ci.ipynb"))
    print("\n=== checks.py ===\n" + open(R + "/shared/src/checks.py").read())
    t = open(R + "/shared/src/rq3.py").read(); tree = ast.parse(t)
    print("\n=== rq3.py: module-level functions (signature | docstring first line) ===")
    for n in tree.body:
        if isinstance(n, ast.FunctionDef):
            print("  def %s(%s) | %s" % (n.name, ", ".join(a.arg for a in n.args.args), (ast.get_docstring(n) or "").splitlines()[0:1]))
    print("\n=== rq3.py: source of the run/driver function(s) (first 90 lines each) ===")
    for n in tree.body:
        if isinstance(n, ast.FunctionDef) and re.search(r"run|config|seed|one", n.name, re.I):
            seg = ast.get_source_segment(t, n).splitlines(); print("\n--- %s (%d lines) ---" % (n.name, len(seg))); print("\n".join(seg[:90]))
    pt = json.load(open(R + "/selection/tables/sel_rq3_pilot_timing.json")); print("\npilot timing extrapolation:", pt.get("extrapolation"), "| timing:", pt.get("timing"))
    ps = pd.read_csv(R + "/selection/tables/sel_rq3_per_seed_v2.csv"); print("selection per-seed run_seconds: sum %.0f s over %d rows (BN rows only: %.0f s)" % (ps.run_seconds.sum(), len(ps), ps[ps.model == "BN"].run_seconds.sum()))
    print("per-seed model/variant values:", sorted(set(zip(ps.model, ps.variant))))
    nb = json.load(open(R + "/selection/notebooks/00_precheck.ipynb")); s3 = "".join([c for c in nb["cells"] if c["cell_type"] == "code"][2]["source"])
    print("\n=== 00_precheck cell 3 (build_design) in full ===\n" + s3)
    def grid_lines(path, label):
        nbk = json.load(open(R + "/" + path)); print("\n=== grid definitions in %s ===" % label)
        for i, c in enumerate([c for c in nbk["cells"] if c["cell_type"] == "code"], 1):
            s = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
            if re.search(r"param_grid|GridSearchCV\(", s):
                ls = s.splitlines(); idx = [j for j, l in enumerate(ls) if re.search(r"grid|GridSearchCV|solver|penalty|\bC\b|n_estimators|max_depth|min_samples_leaf|StratifiedKFold", l)]
                keep = sorted(set(k for j in idx[:40] for k in (j - 1, j, j + 1) if 0 <= k < len(ls))); print("-- code cell %d --" % i); print("\n".join(ls[k][:160] for k in keep[:45]))
    grid_lines("full_variable/notebooks/Expirements_FROZEN.ipynb", "FROZEN (monitoring)"); grid_lines("selection/notebooks/04_baselines_rq2.ipynb", "04 (selection)")
    for f in ("df_train_imputed", "df_test_imputed"):
        d = pd.read_pickle(R + "/Data/processed/%s.pkl" % f); print("\n%s: shape %s | columns %s | dtypes %s | index head %s" % (f, d.shape, list(d.columns), dict(d.dtypes.astype(str)), d.index[:3].tolist()))
    print("\nselection notebooks:", sorted(x for x in os.listdir(R + "/selection/notebooks") if x.endswith(".ipynb")))
_c222()

In [ ]:
# Cell 223 (10_continuous_ref_and_rq3_ci.ipynb, cell 2): Phase 8 inspection 2. READ-ONLY, writes NOTHING.
def _c223():
    import os, sys, ast, json, pickle
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0"
    L = lambda p: pickle.load(open(R + "/" + p, "rb"))
    print("=== full_variable/predictions ===")
    for f in sorted(os.listdir(R + "/full_variable/predictions")):
        o = L("full_variable/predictions/" + f); print(f, type(o).__name__)
        if isinstance(o, dict):
            for k, v in o.items(): print("   %-28s %s" % (k, (v.shape if hasattr(v, "shape") else type(v).__name__ + (" len %d" % len(v) if hasattr(v, "__len__") else ""))))
    sys.path.insert(0, R + "/shared/src")
    t = open(R + "/shared/src/rq3.py").read(); tree = ast.parse(t)
    for n in tree.body:
        if isinstance(n, ast.FunctionDef) and n.name in ("load_context", "validate_0pct", "_row"):
            print("\n--- rq3.%s ---" % n.name); print("\n".join(ast.get_source_segment(t, n).splitlines()[:70]))
    nb = json.load(open(R + "/selection/notebooks/07_rq3_grid.ipynb")); code = [c for c in nb["cells"] if c["cell_type"] == "code"]
    for k in (5,):
        s = "".join(code[k - 1]["source"]); print("\n=== 07 code cell %d (first 120 lines) ===" % k); print("\n".join(s.splitlines()[:120]))
    import rq3
    ctx = rq3.load_context("selection")
    print("\nctx keys:", list(ctx.keys())); print("p0 keys:", list(ctx["p0"].keys()), "| y", np.shape(ctx["y"]), "| models:", list(ctx["models"].keys()), "| base keys:", list(ctx["base"].keys()))
    print("name:", ctx["name"])
_c223()

In [ ]:
# Cell 224 (10_continuous_ref_and_rq3_ci.ipynb, cell 3): Part A0 validation + timing probe. READ-ONLY, writes NOTHING.
def _c224():
    import os, sys, time, hashlib, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    G = {"masking": "c6094c2b232e", "rq3": "9ccbba66073b", "safe_save": "afff88e2b30a", "checks": "ef542907c037", "decision_value": "c1476cef61d1",
         "loglik": "cdc26c96a8d6", "em": "c245161bebeb", "sensitivity": "e49b368ee726", "structure": "effc1d6565f8"}
    for m, h in G.items(): assert hashlib.sha256(open(R + "/shared/src/%s.py" % m, "rb").read()).hexdigest()[:12] == h, "guard mismatch %s: STOP" % m
    sys.path.insert(0, R + "/shared/src"); import rq3, checks
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    L = lambda p: pd.read_pickle(R + "/" + p)
    for stage in ("selection", "monitoring"):
        print("\n######## %s ########" % stage.upper())
        ctx = rq3.load_context(stage); g, base = ctx["g"], ctx["base"]; cols = list(base["imp_tr"].columns)
        tr, te = base["imp_tr"], base["imp_te"]
        if stage == "monitoring":
            A, B = L("Data/processed/df_train_imputed.pkl"), L("Data/processed/df_test_imputed.pkl")
            print("saved imputed frames index equal to rebuilt:", A.index.equals(tr.index), B.index.equals(te.index))
            dm = {c: (float(np.abs(A[c].values - tr[c].values).max()), float(np.abs(B[c].values - te[c].values).max())) for c in cols}
            print("max |saved imputed - rebuilt imputed| per column (train, test):", {c: ("%.2e" % a, "%.2e" % b) for c, (a, b) in dm.items()})
            tr, te = A[cols].copy(), B[cols].copy()
        des = [c for c in g["DESIGN_ORDER"] if c in cols]
        def binned(df):
            out = pd.DataFrame(index=df.index)
            for c in des: out[c + " (bin)"] = g["cut_edges"](df[c], g["EDGES"][c]) if c in g["EDGE_VARS"] else g["bin_natural_category"](df[c], 3)
            return out.astype(str)
        enc = base["encoder"]; Xtr = enc.transform(binned(tr)); Xte = enc.transform(binned(te))
        if stage == "monitoring": Ltr, Lte = L("Data/processed/X_train_df.pkl").values, L("Data/processed/X_test_df.pkl").values
        else: Ltr, Lte = base["Xtr"].values, base["Xte"].values
        ok_bin = np.array_equal(Xtr, Ltr) and np.array_equal(Xte, Lte)
        print("re-binned imputed values == saved binned design (train and test):", ok_bin, "| shapes", Xtr.shape, Xte.shape)
        ok_pred = True
        for k in ("LR", "RF"):
            p = ctx["models"][k].predict_proba(rq3._xin(ctx["models"][k], pd.DataFrame(Xte, columns=base["Xte"].columns, index=te.index)))[:, 1]
            d = float(np.abs(p - ctx["p0"][k]).max()); ok_pred &= d == 0.0; print("  saved binned %s on re-binned design vs saved predictions: max |diff| %.3e" % (k, d))
        if not (ok_bin and ok_pred): print("A0 VALIDATION FAILED for %s: STOP this stage, report, no timing probe" % stage); continue
        print("A0 VALIDATION PASSED for %s. Continuous columns (%d): %s" % (stage, len(cols), cols))
        Xc, ytr = tr.values, np.asarray(ctx["g"]["y_train"])
        t0 = time.time(); rf = RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=1).fit(Xc, ytr); T = time.time() - t0
        print("timing probe RF 500/15/5 on %d x %d continuous: %.1f s per fit -> grid upper bound (27 configs x 5 folds, 0.8 train share, + refit) about %.0f min" % (Xc.shape[0], Xc.shape[1], T, (5 * 9 * 1.6 * 0.8 * T + T) / 60))
        t0 = time.time(); Z = StandardScaler().fit(Xc).transform(Xc)
        lr = checks.fit_converged(LogisticRegression(C=100, solver="lbfgs", penalty="l2", class_weight="balanced", max_iter=5000, random_state=42), Z, ytr, name="LR V1 probe")
        print("timing probe LR V1 (C=100, lbfgs): %.1f s, n_iter_ %d (converged)" % (time.time() - t0, int(lr.n_iter_.max())))
    print("\nDONE (read-only).")
_c224()

In [ ]:
# Cell 225 (10_continuous_ref_and_rq3_ci.ipynb, cell 4): Part B0 fast_auc.py. Validates in memory, then writes (exclusive create, chmod 444) only if ALL pass.
FAST_AUC_SRC = '''"""Weighted AUC for many bootstrap resamples from one sort.
Resamples are multinomial row counts used as weights. Ties use the 0.5 convention (AUC = P(s+ > s-) + 0.5 P(s+ = s-)),
so scores with few distinct values (e.g. a BN with 25 values) are correct. prepare() once per score vector; auc_weighted() per batch of weights."""
import numpy as np


def prepare(scores, y):
    s = np.asarray(scores, float); y = np.asarray(y).astype(int)
    order = np.argsort(s, kind="mergesort"); ss = s[order]
    starts = np.flatnonzero(np.r_[True, ss[1:] != ss[:-1]])
    return dict(order=order, y=y[order].astype(float), starts=starts)


def auc_weighted(prep, W, chunk=200):
    W = np.atleast_2d(np.asarray(W, float)); out = np.empty(W.shape[0]); o, yy, st = prep["order"], prep["y"], prep["starts"]
    for a in range(0, W.shape[0], chunk):
        Wo = W[a:a + chunk][:, o]
        Wp = np.add.reduceat(Wo * yy, st, axis=1); Wn = np.add.reduceat(Wo * (1.0 - yy), st, axis=1)
        cumneg = np.cumsum(Wn, axis=1) - Wn
        num = (Wp * cumneg).sum(1) + 0.5 * (Wp * Wn).sum(1)
        out[a:a + chunk] = num / (Wp.sum(1) * Wn.sum(1))
    return out
'''
def _c225():
    import os, sys, hashlib, warnings
    import numpy as np
    from sklearn.metrics import roc_auc_score
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    G = {"masking": "c6094c2b232e", "rq3": "9ccbba66073b", "safe_save": "afff88e2b30a", "checks": "ef542907c037", "decision_value": "c1476cef61d1",
         "loglik": "cdc26c96a8d6", "em": "c245161bebeb", "sensitivity": "e49b368ee726", "structure": "effc1d6565f8"}
    for m, h in G.items(): assert hashlib.sha256(open(R + "/shared/src/%s.py" % m, "rb").read()).hexdigest()[:12] == h, "guard mismatch %s: STOP" % m
    P = R + "/shared/src/fast_auc.py"; assert not os.path.exists(P), "fast_auc.py exists: STOP"
    ns = {}; exec(compile(FAST_AUC_SRC, "fast_auc", "exec"), ns); prepare, auc_w = ns["prepare"], ns["auc_weighted"]
    sys.path.insert(0, R + "/shared/src"); import rq3
    ctx = rq3.load_context("selection"); y = ctx["y"]; n = len(y)
    r = rq3.run_config(ctx, "Age", 0.2, "mcar", 42, return_preds=True, bp_check=False); valid = r["valid"]
    sets = [("0%% %s" % k, ctx["p0"][k], np.isfinite(ctx["p0"][k])) for k in ("BN", "LR", "RF")] + [("masked Age/mcar/0.20/42 %s" % k, r["preds"][k], valid) for k in ("BN", "LR_primary", "RF_primary")]
    rng = np.random.default_rng(42); worst = 0.0
    for nm, s, v in sets:
        vi = np.flatnonzero(v); pr = prepare(s, y); d0 = abs(float(auc_w(pr, v.astype(float))[0]) - roc_auc_score(y[v], s[v]))
        dd = []
        for _ in range(20):
            idx = rng.choice(vi, size=len(vi), replace=True); W = np.bincount(idx, minlength=n).astype(float)
            dd.append(abs(float(auc_w(pr, W)[0]) - roc_auc_score(y[idx], s[idx])))
        m = max([d0] + dd); worst = max(worst, m); print("  %-34s distinct scores %5d | original diff %.2e | 20 resamples max diff %.2e" % (nm, len(np.unique(s[v])), d0, max(dd)))
    print("worst difference vs sklearn: %.3e" % worst)
    if worst > 1e-12: print("B0 VALIDATION FAILED: STOP, module not written, report"); return
    with open(P, "x") as f: f.write(FAST_AUC_SRC)
    os.chmod(P, 0o444); print("B0 PASSED. Wrote shared/src/fast_auc.py (chmod 444) sha12 %s" % hashlib.sha256(open(P, "rb").read()).hexdigest()[:12])
_c225()

In [ ]:
# Cell 226 (10_continuous_ref_and_rq3_ci.ipynb, cell 5): Part A routine; runs SELECTION. Saves 3 files at the end (exclusive create) only if all validations pass.
def _partA(stage):
    import os, sys, time, json, hashlib, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    G = {"masking": "c6094c2b232e", "rq3": "9ccbba66073b", "safe_save": "afff88e2b30a", "checks": "ef542907c037", "decision_value": "c1476cef61d1",
         "loglik": "cdc26c96a8d6", "em": "c245161bebeb", "sensitivity": "e49b368ee726", "structure": "effc1d6565f8"}
    for m, h in G.items(): assert hashlib.sha256(open(R + "/shared/src/%s.py" % m, "rb").read()).hexdigest()[:12] == h, "guard mismatch %s: STOP" % m
    sys.path.insert(0, R + "/shared/src"); import rq3, checks, safe_save as ss
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold, GridSearchCV
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy.stats import ks_2samp
    pre, folder = ("sel_", "selection/tables/") if stage == "selection" else ("mon_", "monitoring_rq3/")
    outs = [folder + pre + "continuous_reference_" + s for s in ("table.csv", "bootstrap.csv", "notes.json")]
    for o in outs: ss.check_path(os.path.join(R, o))
    L = lambda p: pd.read_pickle(R + "/" + p)
    print("######## %s ########" % stage.upper())
    ctx = rq3.load_context(stage); g, base = ctx["g"], ctx["base"]; cols = list(base["imp_tr"].columns); tr, te = base["imp_tr"], base["imp_te"]
    if stage == "monitoring":
        A, B = L("Data/processed/df_train_imputed.pkl"), L("Data/processed/df_test_imputed.pkl")
        assert A.index.equals(tr.index) and B.index.equals(te.index) and all(np.array_equal(A[c].values, tr[c].values) and np.array_equal(B[c].values, te[c].values) for c in cols), "saved imputed != rebuilt: STOP"
        tr, te = A[cols].copy(), B[cols].copy()
    des = [c for c in g["DESIGN_ORDER"] if c in cols]
    def binned(df):
        out = pd.DataFrame(index=df.index)
        for c in des: out[c + " (bin)"] = g["cut_edges"](df[c], g["EDGES"][c]) if c in g["EDGE_VARS"] else g["bin_natural_category"](df[c], 3)
        return out.astype(str)
    Xtr, Xte = base["encoder"].transform(binned(tr)), base["encoder"].transform(binned(te))
    Ltr, Lte = (L("Data/processed/X_train_df.pkl").values, L("Data/processed/X_test_df.pkl").values) if stage == "monitoring" else (base["Xtr"].values, base["Xte"].values)
    assert np.array_equal(Xtr, Ltr) and np.array_equal(Xte, Lte), "A0: re-binned design != saved design: STOP"
    a0 = {}
    for k in ("LR", "RF"):
        p = ctx["models"][k].predict_proba(rq3._xin(ctx["models"][k], pd.DataFrame(Xte, columns=base["Xte"].columns, index=te.index)))[:, 1]
        a0[k] = float(np.abs(p - ctx["p0"][k]).max()); assert a0[k] <= 1e-12, "A0: saved binned %s does not reproduce saved predictions (%.3e): STOP" % (k, a0[k])
    print("A0 PASSED (designs exact; binned LR/RF vs saved predictions max |diff| LR %.2e RF %.2e, tolerance 1e-12). Continuous columns (%d): %s" % (a0["LR"], a0["RF"], len(cols), cols))
    ytr, y = np.asarray(g["y_train"]), ctx["y"]; n = len(y); P0 = ctx["p0"]; valid = np.isfinite(P0["BN"]); Xc, Xct = tr.values, te.values
    cv = StratifiedKFold(5, shuffle=True, random_state=42)
    # timing probe and estimate
    t0 = time.time(); RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=1).fit(Xc, ytr); T = time.time() - t0
    est = (57.6 * T + T) / 60; print("timing probe RF 500/15/5: %.1f s per fit -> full RF grid upper bound %.0f min" % (T, est))
    if stage == "monitoring" and est > 120: print("ESTIMATE EXCEEDS 2 HOURS: STOP, grid not started, report"); return
    # RF grid
    RFG = {"n_estimators": [100, 200, 500], "max_depth": [5, 10, 15], "min_samples_leaf": [5, 10, 20]}
    t0 = time.time(); gs = GridSearchCV(RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1), RFG, scoring="roc_auc", cv=cv, n_jobs=1).fit(Xc, ytr); t_rf = time.time() - t0
    edge = {k: ("min edge" if gs.best_params_[k] == v[0] else "max edge" if gs.best_params_[k] == v[-1] else "interior") for k, v in RFG.items()}
    p_rf = gs.best_estimator_.predict_proba(Xct)[:, 1]; print("RF grid %.1f min | best %s | CV AUC %.4f | edge flags %s" % (t_rf / 60, gs.best_params_, gs.best_score_, edge))
    # LR variants
    LOGC = ['Debt Ratio', 'Monthly Income', 'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents', '30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due']
    def v2(df):
        d = df.copy()
        for c in LOGC:
            if c in d.columns: d[c] = np.log1p(np.clip(d[c].values, 0, None))
        return d
    CG = [0.01, 0.1, 1, 10, 100]; mk = lambda C: LogisticRegression(C=C, solver="lbfgs", penalty="l2", class_weight="balanced", max_iter=5000, random_state=42)
    VAR = {"V1": (tr.values, te.values), "V2": (v2(tr).values, v2(te).values)}; cvres = {}; fin = {}; t0 = time.time()
    for vn, (Xa, Xb) in VAR.items():
        cvres[vn] = {}
        for C in CG:
            sc = []
            for a, b in cv.split(Xa, ytr):
                s = StandardScaler().fit(Xa[a]); m = checks.fit_converged(mk(C), s.transform(Xa[a]), ytr[a], name="LR %s C=%s" % (vn, C)); sc.append(roc_auc_score(ytr[b], m.predict_proba(s.transform(Xa[b]))[:, 1]))
            cvres[vn][C] = float(np.mean(sc))
        bC = max(cvres[vn], key=cvres[vn].get); s = StandardScaler().fit(Xa); m = checks.fit_converged(mk(bC), s.transform(Xa), ytr, name="LR %s final" % vn)
        fin[vn] = dict(C=bC, cv=cvres[vn][bC], p=m.predict_proba(s.transform(Xb))[:, 1], n_iter=int(m.n_iter_.max()))
        print("LR %s: CV AUC by C %s | best C %s CV %.4f | n_iter %d" % (vn, {c: round(v, 4) for c, v in cvres[vn].items()}, bC, fin[vn]["cv"], fin[vn]["n_iter"]))
    sel = max(fin, key=lambda v: fin[v]["cv"]); other = [v for v in fin if v != sel][0]; print("LR variant selected by train CV AUC: %s (%.4f vs %.4f)" % (sel, fin[sel]["cv"], fin[other]["cv"]))
    P = {"BN": P0["BN"], "LR_binned": P0["LR"], "RF_binned": P0["RF"], "LR_cont": fin[sel]["p"], "RF_cont": p_rf}
    # bootstrap
    sets = {"all": (np.ones(n, bool), ["LR_binned", "RF_binned", "LR_cont", "RF_cont"]), "bn_valid": (valid, ["BN", "LR_binned", "RF_binned", "LR_cont", "RF_cont"])}
    NB = 2000; rng = np.random.default_rng(42); BA = {(s, m): np.empty(NB) for s, (_, ms) in sets.items() for m in ms}; t0 = time.time()
    for b in range(NB):
        idx = rng.integers(0, n, n); idv = idx[valid[idx]]
        for s, (_, ms) in sets.items():
            ii = idx if s == "all" else idv
            for m in ms: BA[(s, m)][b] = roc_auc_score(y[ii], P[m][ii])
    print("bootstrap %d resamples: %.1f min" % (NB, (time.time() - t0) / 60))
    def met(p, v):
        yy, pp = y[v], p[v]; a = roc_auc_score(yy, pp); return a, 2 * a - 1, float(ks_2samp(pp[yy == 1], pp[yy == 0]).statistic), float(average_precision_score(yy, pp))
    rows = []
    for s, (v, ms) in sets.items():
        for m in ms:
            a, gi, ks, ap = met(P[m], v); lo, hi = np.percentile(BA[(s, m)], [2.5, 97.5]); rows.append(dict(model=m, role="selected variant " + sel if m == "LR_cont" else "", row_set=s, n=int(v.sum()), auc=a, auc_ci_lo=lo, auc_ci_hi=hi, gini=gi, ks=ks, ap=ap))
        if s == "all":
            a, gi, ks, ap = met(fin[other]["p"], v); rows.append(dict(model="LR_cont_%s" % other, role="not used for selection", row_set=s, n=int(v.sum()), auc=a, auc_ci_lo=np.nan, auc_ci_hi=np.nan, gini=gi, ks=ks, ap=ap))
    for vv in (sets["bn_valid"][0],):
        a, gi, ks, ap = met(fin[other]["p"], vv); rows.append(dict(model="LR_cont_%s" % other, role="not used for selection", row_set="bn_valid", n=int(vv.sum()), auc=a, auc_ci_lo=np.nan, auc_ci_hi=np.nan, gini=gi, ks=ks, ap=ap))
    tab = pd.DataFrame(rows)
    def classify(d, lo, hi): return "equal" if abs(d) < 0.002 else ("continuous higher" if d > 0.002 and lo > 0 else "continuous lower" if d < -0.002 and hi < 0 else "inconclusive")
    brow = []
    for nm, (s, m1, m2) in {"continuous LR minus binned LR": ("all", "LR_cont", "LR_binned"), "continuous RF minus binned RF": ("all", "RF_cont", "RF_binned"),
                            "continuous LR minus BN": ("bn_valid", "LR_cont", "BN"), "continuous RF minus BN": ("bn_valid", "RF_cont", "BN")}.items():
        v = sets[s][0]; d = roc_auc_score(y[v], P[m1][v]) - roc_auc_score(y[v], P[m2][v]); dist = BA[(s, m1)] - BA[(s, m2)]; lo, hi = np.percentile(dist, [2.5, 97.5])
        brow.append(dict(comparison=nm, row_set=s, diff=d, ci_lo=lo, ci_hi=hi, n_resamples=NB, classification=classify(d, lo, hi) if "binned" in nm else "(not classified)"))
    bt = pd.DataFrame(brow)
    notes = dict(stage=stage, written="2026-10-17 (Phase 8 Part A)", a0=dict(saved_binned_pred_max_abs_diff=a0, tolerance=1e-12, design_equality="exact"), continuous_columns=cols,
                 rf=dict(grid=RFG, best_params=gs.best_params_, cv_auc=float(gs.best_score_), grid_edge_flags=edge, minutes=t_rf / 60),
                 lr=dict(C_grid=CG, solver="lbfgs", penalty="l2", max_iter=5000, variants_cv_auc_by_C={v: {str(c): x for c, x in d.items()} for v, d in cvres.items()}, selected_variant=sel, selected_C=fin[sel]["C"],
                         selected_cv_auc=fin[sel]["cv"], other_variant=other, other_C=fin[other]["C"], other_cv_auc=fin[other]["cv"], variant_selection_rule="5-fold CV AUC on train only; scaler fit inside each fold",
                         note="The binned LR grid used saga with l1/l2; the continuous LR uses lbfgs/l2 only because l1 needs saga and both binned winners were l2.",
                         v2_transform="log1p(clip(x,0)) on " + ", ".join(c for c in LOGC if c in cols) + ", then StandardScaler (train-only)"),
                 grids_source="LR C grid and RF restricted grid as printed from frozen notebook cells 37/40 and 04 cell 7 (Cell 222)", row_counts=dict(all=n, bn_valid=int(valid.sum())), bootstrap=dict(resamples=NB, seed=42, same_resamples_all_comparisons=True),
                 classification_rule="equal if |diff|<0.002; continuous higher if >+0.002 and CI excludes 0; continuous lower if <-0.002 and CI excludes 0; else inconclusive", classification={r["comparison"]: r["classification"] for r in brow},
                 no_thresholds_or_costs_derived=True, imputed_dependents="used as-is (no int() truncation)", runtime_min=dict(rf_grid=t_rf / 60))
    print("\nTABLE"); print(tab.round(4).to_string(index=False)); print("\nBOOTSTRAP"); print(bt.round(4).to_string(index=False))
    for rel, data in ((outs[0], tab.to_csv(index=False)), (outs[1], bt.to_csv(index=False)), (outs[2], json.dumps(notes, indent=1, default=str))):
        with open(ss.check_path(os.path.join(R, rel)), "x") as f: f.write(data)
        print("saved", rel, hashlib.sha256(open(R + "/" + rel, "rb").read()).hexdigest()[:12])
# _partA("selection")

In [ ]:
# Cell 227 (10_continuous_ref_and_rq3_ci.ipynb, cell 6): Part A MONITORING. Same kernel as Cell 226 (needs _partA).
_partA("monitoring")

In [ ]:
# Cell 228 (10_continuous_ref_and_rq3_ci.ipynb): Part B1. Selection RQ3 grid with predictions SAVED + reproduction check. Saves sel_rq3_masked_predictions.pkl only if ALL checks pass.
def _b1():
    import os, sys, time, json, pickle, hashlib, warnings
    import numpy as np, pandas as pd
    from sklearn.metrics import roc_auc_score
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    G = {"masking": "c6094c2b232e", "rq3": "9ccbba66073b", "safe_save": "afff88e2b30a", "checks": "ef542907c037", "decision_value": "c1476cef61d1",
         "loglik": "cdc26c96a8d6", "em": "c245161bebeb", "sensitivity": "e49b368ee726", "structure": "effc1d6565f8"}
    for m, h in G.items(): assert hashlib.sha256(open(R + "/shared/src/%s.py" % m, "rb").read()).hexdigest()[:12] == h, "guard mismatch %s: STOP" % m
    assert hashlib.sha256(open(R + "/shared/src/fast_auc.py", "rb").read()).hexdigest()[:12] == "66200c06261d", "fast_auc changed: STOP"
    sys.path.insert(0, R + "/shared/src"); import rq3, safe_save as ss
    OUT = "selection/sel_rq3_masked_predictions.pkl"; ss.check_path(os.path.join(R, OUT))
    ctx = rq3.load_context("selection")
    th = json.load(open(R + "/selection/tables/sel_rq3_thresholds_r10.json"))["taus"]; ctx["tau"] = {k: float(th[k]) for k in ("BN", "LR", "RF")}
    ps = pd.read_csv(R + "/selection/tables/sel_rq3_per_seed_v2.csv"); ps["level"] = ps["level"].round(2)
    ref = ps.set_index(["variable", "level", "mechanism", "seed", "variant"])
    y, p0 = ctx["y"], ctx["p0"]; saved = {}; worst = 0.0; t_all = time.time(); nrun = 0
    LEVELS, SEEDS, MECHS = (0.20, 0.40, 0.50), range(42, 52), ("mcar", "dependent")
    for var in rq3.MASK_VARS["selection"]:
        for mech in MECHS:
            for lev in LEVELS:
                t_c = time.time()
                for seed in SEEDS:
                    r = rq3.run_config(ctx, var, lev, mech, seed, secondary="median", bp_check=(seed == 42), return_preds=True, bin_mode="wrap")
                    v = r["valid"]; key = (var, round(lev, 2), mech, seed)
                    for vname, pk in (("BN", "BN"), ("LR_primary", "LR_primary"), ("RF_primary", "RF_primary")):
                        mdl = vname.split("_")[0]; p = r["preds"][pk]
                        a = roc_auc_score(y[v], p[v]); d = a - roc_auc_score(y[v], p0[mdl][v]); rr = ref.loc[key + (vname,)]
                        for nm, val in (("auc", a), ("delta_auc", d)):
                            diff = abs(val - float(rr[nm])); worst = max(worst, diff)
                            assert diff <= 1e-12, "B1 REPRODUCTION FAILED: %s %s %s diff %.3e vs per-seed file: STOP, report" % (key, vname, nm, diff)
                        assert int(rr["n_valid"]) == int(v.sum()), "n_valid mismatch %s: STOP" % (key,)
                    saved[key] = dict(BN=r["preds"]["BN"].copy(), LR=r["preds"]["LR_primary"].copy(), RF=r["preds"]["RF_primary"].copy(), valid=v.copy(), mask=r["mask"].copy(), realised=r["realised"]); nrun += 1
                print("  %-12s %-9s %.2f | %d seeds reproduced | %.0fs | running worst diff %.2e" % (var, mech, lev, len(SEEDS), time.time() - t_c, worst), flush=True)
    sm = pd.read_csv(R + "/selection/tables/sel_rq3_summary_v2.csv"); sm["level"] = sm["level"].round(2); worst_s = 0.0
    for (var, lev, mech), _ in sm.groupby(["variable", "level", "mechanism"]):
        for vname, mdl in (("BN", "BN"), ("LR_primary", "LR"), ("RF_primary", "RF")):
            ds = []
            for seed in SEEDS:
                s = saved[(var, lev, mech, seed)]; v = s["valid"]; ds.append(roc_auc_score(y[v], s[mdl][v]) - roc_auc_score(y[v], p0[mdl][v]))
            ref_m = float(sm[(sm.variable == var) & (sm.level == lev) & (sm.mechanism == mech) & (sm.variant == vname)].mean_delta_auc.iloc[0])
            worst_s = max(worst_s, abs(np.mean(ds) - ref_m)); assert abs(np.mean(ds) - ref_m) <= 1e-12, "summary mismatch %s %s %s %s: STOP" % (var, lev, mech, vname)
    print("\nB1 REPRODUCTION PASSED: %d seed runs; per-seed AUC and delta AUC max diff %.2e; summary mean delta AUC (BN, LR_primary, RF_primary) max diff %.2e; %.1f min" % (nrun, worst, worst_s, (time.time() - t_all) / 60))
    blob = dict(y=y, row_labels=np.asarray(ctx["test_idx"]), p0={k: np.asarray(v) for k, v in p0.items()}, runs=saved,
                note="LARGE INTERMEDIATE: excluded from submission. Primary imputer only; BN, LR_primary, RF_primary per-row test predictions; valid = rows with defined BN posterior at 0% and in the config.")
    with open(ss.check_path(os.path.join(R, OUT)), "xb") as f: pickle.dump(blob, f, protocol=4)
    print("saved", OUT, os.path.getsize(R + "/" + OUT) // 1_000_000, "MB sha12", hashlib.sha256(open(R + "/" + OUT, "rb").read()).hexdigest()[:12])
_b1()

In [ ]:
# Cell 229 (10_continuous_ref_and_rq3_ci.ipynb): Part B2 test-row bootstrap of paired BN-minus-LR/RF delta AUC. Saves 3 files only if all checks pass.
def _b2():
    import os, sys, io, time, json, pickle, hashlib, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    G = {"masking": "c6094c2b232e", "rq3": "9ccbba66073b", "safe_save": "afff88e2b30a", "checks": "ef542907c037", "decision_value": "c1476cef61d1",
         "loglik": "cdc26c96a8d6", "em": "c245161bebeb", "sensitivity": "e49b368ee726", "structure": "effc1d6565f8", "fast_auc": "66200c06261d"}
    for m, h in G.items(): assert hashlib.sha256(open(R + "/shared/src/%s.py" % m, "rb").read()).hexdigest()[:12] == h, "guard mismatch %s: STOP" % m
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss, fast_auc as fa
    O = {"csv": "selection/tables/sel_rq3_row_bootstrap_summary.csv", "json": "selection/tables/sel_rq3_row_bootstrap_notes.json", "png": "selection/figures/sel_rq3_paired_ci.png"}
    for o in O.values(): ss.check_path(os.path.join(R, o))
    blob = pickle.load(open(R + "/selection/sel_rq3_masked_predictions.pkl", "rb")); y, p0, runs = blob["y"], blob["p0"], blob["runs"]; n = len(y)
    sm = pd.read_csv(R + "/selection/tables/sel_rq3_summary_v2.csv"); sm["level"] = sm["level"].round(2)
    B = 2000; rng = np.random.default_rng(42); W = rng.multinomial(n, np.full(n, 1.0 / n), size=B).astype(np.int16)
    MODELS = ("BN", "LR", "RF"); fix = lambda p: np.nan_to_num(np.asarray(p, float), nan=0.0)
    P0 = {m: fa.prepare(fix(p0[m]), y) for m in MODELS}
    def batch(prep, v):
        out = np.empty(B)
        for a in range(0, B, 500): out[a:a + 500] = fa.auc_weighted(prep, W[a:a + 500].astype(float) * v, chunk=500)
        return out
    k0 = next(iter(runs)); v0 = runs[k0]["valid"].astype(float); t0 = time.time(); batch(P0["BN"], v0); tb = time.time() - t0
    est = tb * 12 * 10 * 3 * 2; print("one 2000-resample batch: %.2f s -> estimated total %.1f min" % (tb, est / 60))
    if est > 7200: print("ESTIMATE EXCEEDS 2 HOURS: STOP, not started"); return
    keys = sorted({(k[0], k[1], k[2]) for k in runs}); rows = []; t_all = time.time(); worst = 0.0
    base_cache = {}
    for (var, lev, mech) in keys:
        Dn = {m: np.zeros(B) for m in MODELS}; Dp = {m: 0.0 for m in MODELS}; ns = 0
        for seed in range(42, 52):
            r = runs[(var, lev, mech, seed)]; v = r["valid"].astype(float); ns += 1
            for m in MODELS:
                pm = fa.prepare(fix(r[m]), y)
                a_b = batch(pm, v); b_b = batch(P0[m], v); Dn[m] += a_b - b_b
                Dp[m] += float(fa.auc_weighted(pm, v)[0] - fa.auc_weighted(P0[m], v)[0])
        assert ns == 10
        for m in MODELS: Dn[m] /= ns; Dp[m] /= ns
        for other, vn in (("LR", "LR_primary"), ("RF", "RF_primary")):
            ref = lambda var_nm: float(sm[(sm.variable == var) & (sm.level == lev) & (sm.mechanism == mech) & (sm.variant == var_nm)].mean_delta_auc.iloc[0])
            ref_c = ref("BN") - ref(vn); pt = Dp["BN"] - Dp[other]; worst = max(worst, abs(pt - ref_c))
            assert abs(pt - ref_c) <= 1e-12, "POINT ESTIMATE MISMATCH %s %s %s BN-%s: %.3e vs %.3e: STOP, report" % (var, lev, mech, other, pt, ref_c)
            C = Dn["BN"] - Dn[other]; lo, hi = np.percentile(C, [2.5, 97.5]); reading = "BN degrades less" if lo > 0 else ("BN degrades more" if hi < 0 else "tie")
            rows.append(dict(variable=var, level=lev, mechanism=mech, contrast="BN minus " + other, point_estimate=pt, summary_file_value=ref_c, boot_mean=float(C.mean()), ci_lo=lo, ci_hi=hi,
                             share_gt0=float((C > 0).mean()), reading=reading, n_resamples=B, n_seeds=ns))
        print("  %-11s %-9s %.2f done | %.1f min elapsed" % (var, mech, lev, (time.time() - t_all) / 60), flush=True)
    df = pd.DataFrame(rows); print("\nPOINT ESTIMATES match sel_rq3_summary_v2.csv: max diff %.2e" % worst); print(df.round(4).to_string(index=False))
    # figure
    FOOT = "Positive = the BN degrades less. Intervals reflect test-row sampling and averaging over 10 mask seeds; models fixed. Primary imputer."
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharey=True); col = {"BN minus LR": "#0072B2", "BN minus RF": "#D55E00"}; ls = {"mcar": "-", "dependent": "--"}; mk = {"mcar": "o", "dependent": "s"}; off = {"mcar": -0.008, "dependent": 0.008}
    for ax, var in zip(axes, ("Age", "Debt Ratio")):
        for (ct, mech), d in df[df.variable == var].groupby(["contrast", "mechanism"]):
            d = d.sort_values("level"); x = d.level.values + off[mech] + (0.003 if ct.endswith("RF") else -0.003)
            ax.errorbar(x, d.point_estimate, yerr=[d.point_estimate - d.ci_lo, d.ci_hi - d.point_estimate], color=col[ct], ls=ls[mech], marker=mk[mech], capsize=3, lw=1.6, label="%s, %s" % (ct, "MCAR" if mech == "mcar" else "dependent"))
        ax.axhline(0, color="black", lw=1); ax.set_title(var, fontsize=13); ax.set_xlabel("Masking level"); ax.set_xticks([0.2, 0.4, 0.5]); ax.set_xticklabels(["20%", "40%", "50%"])
    axes[0].set_ylabel("Difference in AUC loss (BN minus model)"); axes[0].legend(fontsize=8, frameon=False, loc="best")
    fig.suptitle("Selection RQ3: paired BN minus LR / RF change in AUC, 95% test-row bootstrap intervals", fontsize=13); fig.text(0.01, 0.005, FOOT, fontsize=8.5, va="bottom"); fig.subplots_adjust(bottom=0.16, top=0.88)
    bf = io.BytesIO(); fig.savefig(bf, format="png", dpi=300); plt.close(fig)
    notes = dict(written="2026-10-18 (Phase 8 Part B2)", resamples=B, seed=42, design="one shared 2000 x 30000 multinomial count matrix used for every seed run, model and configuration; counts zeroed on rows invalid in that seed run; the 0% baseline of that run uses the same restricted counts",
                 statistic="D(m,b) = mean over the 10 mask seeds of [masked AUC - 0% AUC]; contrast C(b) = D(BN,b) - D(model,b); positive = the BN degrades less",
                 point_estimate_check="equals the 10-seed mean paired difference from sel_rq3_summary_v2.csv; max abs diff %.2e (limit 1e-12)" % worst,
                 reading_rule="BN degrades less if the 95% percentile interval excludes 0 and is positive; tie if it includes 0; BN degrades more if it excludes 0 and is negative",
                 coverage="Intervals cover test-row sampling and averaging over mask seeds, NOT retraining variability (models stay fixed). Primary imputer only; the median-imputed secondary models are not included.",
                 readings={r: int(c) for r, c in df.reading.value_counts().items()}, b1_reproduction="all 120 seed runs reproduced sel_rq3_per_seed_v2.csv (max diff 9.97e-17)", fast_auc_sha12=G["fast_auc"],
                 excluded_large_intermediate="selection/sel_rq3_masked_predictions.pkl (94 MB, sha12 e1873f694fcf): excluded from submission, not committed", runtime_min=(time.time() - t_all) / 60)
    for rel, data, mode in ((O["csv"], df.to_csv(index=False), "x"), (O["json"], json.dumps(notes, indent=1), "x"), (O["png"], bf.getvalue(), "xb")):
        with open(ss.check_path(os.path.join(R, rel)), mode) as f: f.write(data)
        print("saved", rel, hashlib.sha256(open(R + "/" + rel, "rb").read()).hexdigest()[:12])
_b2()

In [ ]:
# Cell 230 (10_continuous_ref_and_rq3_ci.ipynb): Part C. Manifest append + ONE commit (files I created + MANIFEST.csv). Tags unmoved. Predictions pickle NOT committed.
def _c230():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0: raise RuntimeError("git %s failed: %s" % (" ".join(a), r.stderr.strip()[:300]))
        return r.stdout.strip()
    sha = lambda p: hashlib.sha256(open(R + "/" + p, "rb").read()).hexdigest()
    assert git("rev-list", "-n", "1", "analysis-final").startswith("6dddbb8") and git("rev-list", "-n", "1", "full-variable-final").startswith("5626192"), "tag moved: STOP"
    mp = R + "/MANIFEST.csv"; raw0 = open(mp, "rb").read(); recs = list(csv.DictReader(open(mp, newline="")))
    assert raw0.endswith(b"\n") and len(recs) == 268, "unexpected manifest state (%d rows): STOP" % len(recs)
    NB = "selection/notebooks/10_continuous_ref_and_rq3_ci.ipynb"; PK = "selection/sel_rq3_masked_predictions.pkl"
    FILES = ["shared/src/fast_auc.py", NB,
             "selection/tables/sel_continuous_reference_table.csv", "selection/tables/sel_continuous_reference_bootstrap.csv", "selection/tables/sel_continuous_reference_notes.json",
             "monitoring_rq3/mon_continuous_reference_table.csv", "monitoring_rq3/mon_continuous_reference_bootstrap.csv", "monitoring_rq3/mon_continuous_reference_notes.json",
             "selection/tables/sel_rq3_row_bootstrap_summary.csv", "selection/tables/sel_rq3_row_bootstrap_notes.json", "selection/figures/sel_rq3_paired_ci.png"]
    have = {r["path"] for r in recs}
    for f in FILES + [PK]: assert os.path.exists(R + "/" + f), "missing %s: STOP" % f
    assert not (set(FILES) | {PK}) & have, "a file is already in the manifest: STOP"
    assert sha("shared/src/fast_auc.py")[:12] == "66200c06261d", "fast_auc changed: STOP"
    now = time.strftime("%Y-%m-%d %H:%M"); NOTE = "added at Phase 8, 2026-10-18"
    ex = {NB: "; notebook (git copy is the reference)", PK: "; LARGE INTERMEDIATE, excluded from submission and not committed"}
    role = lambda p: "shared" if p.startswith("shared/") else "monitoring_rq3" if p.startswith("monitoring_rq3") else "selection"
    rows = [[f, os.path.getsize(R + "/" + f), sha(f), role(f), now, NOTE + ex.get(f, "")] for f in FILES + [PK]]
    with open(mp, "a", newline="") as fh: csv.writer(fh).writerows(rows)
    assert len(list(csv.DictReader(open(mp, newline="")))) == 268 + len(rows) and open(mp, "rb").read().startswith(raw0)
    git("add", "MANIFEST.csv", "shared/src/fast_auc.py", NB); git("add", "-f", *[f for f in FILES if f not in ("shared/src/fast_auc.py", NB)])
    staged = set(git("diff", "--cached", "--name-only").splitlines()); assert staged == set(FILES) | {"MANIFEST.csv"}, "unexpected staged set: %s STOP" % sorted(staged)
    ident = [] if (git("config", "user.name", check=False) and git("config", "user.email", check=False)) else ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 8: continuous-feature reference (both stages), fast_auc module, selection RQ3 test-row bootstrap intervals")
    print("manifest rows now:", 268 + len(rows), "(12 appended; predictions pickle listed, not committed)")
    print("commit:", git("log", "--oneline", "-1")); print("tags unmoved:", git("rev-list", "-n", "1", "analysis-final").startswith("6dddbb8"), git("rev-list", "-n", "1", "full-variable-final").startswith("5626192"))
    print("pickle tracked:", bool(git("ls-files", "--", PK))); print("git status:", git("status", "--porcelain").splitlines())
_c230()